In [11]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/competitions/cifar-10/trainLabels.csv
/kaggle/input/competitions/cifar-10/sampleSubmission.csv
/kaggle/input/competitions/cifar-10/test.7z
/kaggle/input/competitions/cifar-10/train.7z


#1. Understanding CIFAR-10

The dataset consists of **60,000 tiny images** categorized into **10 mutually exclusive classes**:

- airplane  
- automobile  
- bird  
- cat  
- deer  
- dog  
- frog  
- horse  
- ship  
- truck  

---

## Dataset Details

- **Image Size:** `32 × 32` pixels  
- **Channels:** `3` (Red, Green, Blue)  
- **Train/Test Split:**
  - Training Images: **50,000**
  - Testing Images: **10,000**

---

## The Practical Reality

The `32 × 32` resolution is **extremely low**.

- To humans → images may look like **blurry blobs**
- To CNNs → limited spatial information

---

## Key Limitation

You **cannot**:

- Use too many **Pooling layers**
- Use large **Strides** early in the network

---

## Why?

Because:

> You may lose all spatial information before the network learns meaningful features like:
>
> - edges  
> - wings  
> - wheels  

---

## ✅ Key Insight

Design CNN architectures carefully for CIFAR-10:

- Preserve spatial resolution in early layers  
- Avoid aggressive downsampling  
- Extract features gradually  


In [12]:
import os

In [13]:
!apt-get install p7zip-full -y
!7z x /kaggle/input/competitions/cifar-10/train.7z -o./

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
p7zip-full is already the newest version (16.02+dfsg-8).
0 upgraded, 0 newly installed, 0 to remove and 133 not upgraded.

7-Zip [64] 16.02 : Copyright (c) 1999-2016 Igor Pavlov : 2016-05-21
p7zip Version 16.02 (locale=en_US.UTF-8,Utf16=on,HugeFiles=on,64 bits,4 CPUs Intel(R) Xeon(R) CPU @ 2.00GHz (50653),ASM,AES-NI)

Scanning the drive for archives:
  0M Scan /kaggle/input/competitions/cifar-1                                            1 file, 109723070 bytes (105 MiB)

Extracting archive: /kaggle/input/competitions/cifar-10/train.7z
--
Path = /kaggle/input/competitions/cifar-10/train.7z
Type = 7z
Physical Size = 109723070
Headers Size = 294768
Method = LZMA:26
Solid = +
Blocks = 1

        
Would you like to replace the existing file:
  Path:     ./train/1.png
  Size:     2455 bytes (3 KiB)
  Modified: 2013-10-18 17:14:08
with the file from archive:
  Path:     train/1.png
  Size:     245

In [14]:

print(len(os.listdir('./train'))) 
# Should output 50000

50000


In [15]:
import torch
import pandas as pd
import os
import glob
from PIL import Image
from torch.utils.data import Dataset, DataLoader, random_split
import torchvision.transforms as transforms
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim


In [16]:


class CIFAR10Kaggle(Dataset):
    def __init__(self, csv_file, img_dir, transform=None):
        self.df = pd.read_csv(csv_file)
        self.img_dir = img_dir
        self.transform = transform
        self.label_mapping = {
            'airplane':0, 'automobile':1, 'bird':2, 'cat':3, 'deer':4,
            'dog':5, 'frog':6, 'horse':7, 'ship':8, 'truck':9
        }

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        img_id = str(self.df.iloc[idx, 0])
        img_name = os.path.join(self.img_dir, img_id + ".png")
        image = Image.open(img_name)
        label = self.label_mapping[self.df.iloc[idx, 1]]
        if self.transform:
            image = self.transform(image)
        return image, label


train_csv = '/kaggle/input/competitions/cifar-10/trainLabels.csv'
train_dir = './train' 
transform = transforms.Compose([
    transforms.Resize((32, 32)),
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])


full_dataset = CIFAR10Kaggle(csv_file=train_csv, img_dir=train_dir, transform=transform)
train_size = 45000
val_size = 5000
train_data, val_data = random_split(full_dataset, [train_size, val_size])

train_loader = DataLoader(train_data, batch_size=64, shuffle=True)
val_loader = DataLoader(val_data, batch_size=64, shuffle=False)

In [17]:
# Get one batch of data
images, labels = next(iter(train_loader))

print(f"Image batch shape: {images.shape}") # Should be [64, 3, 32, 32]
print(f"Label batch shape: {labels.shape}") # Should be [64]
print(f"First label ID: {labels[0].item()}")

Image batch shape: torch.Size([64, 3, 32, 32])
Label batch shape: torch.Size([64])
First label ID: 4


In [18]:
class KaggleCNN(nn.Module):
    def __init__(self):
        super(KaggleCNN, self).__init__()
        self.conv1 = nn.Conv2d(3, 32, 3, padding=1)
        self.bn1 = nn.BatchNorm2d(32)
        self.conv2 = nn.Conv2d(32, 32, 3, padding=1)
        self.bn2 = nn.BatchNorm2d(32)
        self.conv3 = nn.Conv2d(32, 64, 3, padding=1)
        self.bn3 = nn.BatchNorm2d(64)
        self.pool = nn.MaxPool2d(2, 2)
        self.dropout = nn.Dropout(0.3)
        self.fc1 = nn.Linear(64 * 8 * 8, 128)
        self.fc2 = nn.Linear(128, 10)

    def forward(self, x):
        x = F.relu(self.bn1(self.conv1(x)))
        x = self.pool(F.relu(self.bn2(self.conv2(x))))
        x = self.dropout(x)
        x = self.pool(F.relu(self.bn3(self.conv3(x))))
        x = self.dropout(x)
        x = x.view(-1, 64 * 8 * 8)
        x = F.relu(self.fc1(x))
        x = self.fc2(x)
        return x

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = KaggleCNN().to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

# Training & Accuracy Loop
for epoch in range(10):
    model.train()
    running_loss = 0.0
    for i, (images, labels) in enumerate(train_loader):
        images, labels = images.to(device), labels.to(device)
        
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item()
        if i % 200 == 199:
            print(f'Epoch {epoch+1}, Batch {i+1}: Loss {running_loss / 200:.4f}')
            running_loss = 0.0

    #CALCULATE ACCURACY
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
    
    print(f"--- Epoch {epoch+1} | Validation Accuracy: {100 * correct / total:.2f}% ---")

torch.save(model.state_dict(), "model.pth")

Epoch 1, Batch 200: Loss 1.6207
Epoch 1, Batch 400: Loss 1.3009
Epoch 1, Batch 600: Loss 1.1762
--- Epoch 1 | Validation Accuracy: 53.68% ---
Epoch 2, Batch 200: Loss 1.0251
Epoch 2, Batch 400: Loss 1.0170
Epoch 2, Batch 600: Loss 0.9752
--- Epoch 2 | Validation Accuracy: 67.78% ---
Epoch 3, Batch 200: Loss 0.9053
Epoch 3, Batch 400: Loss 0.8940
Epoch 3, Batch 600: Loss 0.8934
--- Epoch 3 | Validation Accuracy: 68.60% ---
Epoch 4, Batch 200: Loss 0.8357
Epoch 4, Batch 400: Loss 0.8371
Epoch 4, Batch 600: Loss 0.8280
--- Epoch 4 | Validation Accuracy: 71.54% ---
Epoch 5, Batch 200: Loss 0.7808
Epoch 5, Batch 400: Loss 0.7869
Epoch 5, Batch 600: Loss 0.7980
--- Epoch 5 | Validation Accuracy: 72.14% ---
Epoch 6, Batch 200: Loss 0.7428
Epoch 6, Batch 400: Loss 0.7541
Epoch 6, Batch 600: Loss 0.7606
--- Epoch 6 | Validation Accuracy: 73.96% ---
Epoch 7, Batch 200: Loss 0.7077
Epoch 7, Batch 400: Loss 0.7249
Epoch 7, Batch 600: Loss 0.7160
--- Epoch 7 | Validation Accuracy: 74.36% ---
Epoch 

In [19]:
# Create inverse mapping for labels
inv_map = {v: k for k, v in full_dataset.label_mapping.items()}

# Get test files
test_dir = './test'
test_files = glob.glob(os.path.join(test_dir, "*.png"))
test_files.sort(key=lambda x: int(os.path.basename(x).split('.')[0]))

model.eval()
results = []

print("Generating submission.csv...")
with torch.no_grad():
    for path in test_files:
        img_id = os.path.basename(path).split('.')[0]
        img = Image.open(path)
        img = transform(img).unsqueeze(0).to(device) # Batch of 1
        
        output = model(img)
        _, pred = torch.max(output, 1)
        results.append([img_id, inv_map[pred.item()]])

# Save to CSV
df_sub = pd.DataFrame(results, columns=['id', 'label'])
df_sub.to_csv('submission.csv', index=False)


Generating submission.csv...
